In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from einops import einsum
from IPython.display import display
from jaxtyping import Float
from pydantic import ConfigDict
from pydantic.dataclasses import dataclass
from tqdm.auto import tqdm

from src.config.base import ConfigMethodsMixin
from src.data.base import DataConfig
from src.data.dataloading import DataLoaderConfig
from src.data.unity_gaussian_mixture.config import UnityGaussianMixtureDatasetConfig
from src.data.unity_gaussian_mixture.utils import compute_mode_ring_residual
from src.method.drifting.config import (
    DriftingMethodConfig,
    DriftingModelConfig,
    GaussianDriftingLossConfig,
)
from src.model.mlp import StackedResidualMLPConfig
from src.utils import set_seed

SEED = 42

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("seed:", SEED)


In [ ]:
@dataclass(
    kw_only=True,
    config=ConfigDict(
        arbitrary_types_allowed=True,
        extra="forbid",
    ),
)
class ExperimentConfig(ConfigMethodsMixin):
    method_config: DriftingMethodConfig
    lr: float
    train_epochs: int
    log_every_epochs: int
    display_every_epochs: int
    grad_clip: float


dataset_config = UnityGaussianMixtureDatasetConfig(
    num_modes=5,
    ambient_dim=32,
    mode_std=0.1,
    ring_radius_in_zscore=1.5,
    scale=1.0,
    offset_norm=0.0,
    embed_seed=7,
    train_size=2048 * 128,
    val_size=2048 * 16,
)

dataloader_config = DataLoaderConfig(
    batch_size=2048,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True,
)
data_config = DataConfig(
    seed=SEED,
    dataset_config=dataset_config,
    trainloader_config=dataloader_config,
    valloader_config=dataloader_config,
)

decoder_config = StackedResidualMLPConfig.initialize(
    layer_dims=[dataset_config.ambient_dim, 256, 256, 256, dataset_config.ambient_dim],
)

config = ExperimentConfig(
    method_config=DriftingMethodConfig(
        model=DriftingModelConfig(
            decoder_config=decoder_config,
            latent_shape=(dataset_config.ambient_dim,),
        ),
        loss=GaussianDriftingLossConfig(
            objective="forward_kl",
            bandwidth=0.75,
            drift_scale=1.0,
            exclude_self_interactions=True,
            stability_eps=1e-6,
        ),
    ),
    lr=3e-4,
    train_epochs=300,
    log_every_epochs=1,
    display_every_epochs=5,
    grad_clip=1.0,
)
config.visualize()

In [ ]:
def project_to_plane(
    points_hd: Float[torch.Tensor, "batch ambient_dim"],
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    dataset_config: UnityGaussianMixtureDatasetConfig,
) -> Float[torch.Tensor, "batch plane_dim"]:
    displacement = dataset_config.get_displacement().to(
        device=points_hd.device, dtype=points_hd.dtype
    )
    centered = points_hd - displacement
    return (
        einsum(
            centered,
            basis,
            "batch ambient_dim, ambient_dim plane_dim -> batch plane_dim",
        )
        / dataset_config.scale
    )


def orthogonal_residual(
    points_hd: Float[torch.Tensor, "batch ambient_dim"],
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    dataset_config: UnityGaussianMixtureDatasetConfig,
) -> Float[torch.Tensor, "batch ambient_dim"]:
    displacement = dataset_config.get_displacement().to(
        device=points_hd.device, dtype=points_hd.dtype
    )
    projected = project_to_plane(points_hd, basis, dataset_config)
    lifted = einsum(
        projected,
        basis,
        "batch plane_dim, ambient_dim plane_dim -> batch ambient_dim",
    )
    lifted = dataset_config.scale * lifted + displacement
    return points_hd - lifted


basis = data_config.dataset_config.get_basis().to(DEVICE)
train_loader = data_config.get_trainloader()
val_loader = data_config.get_valloader()
preview_batch = next(iter(val_loader)).to(DEVICE)
preview_batch.x_hd.shape, preview_batch.x_2d.shape


In [ ]:
def collect_metrics(
    config: ExperimentConfig,
    data_config: DataConfig,
    val_loader: torch.utils.data.DataLoader,
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    model: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        batch = next(iter(val_loader)).to(device)
        sample_size = int(batch.x_hd.shape[0])
        z = torch.randn(
            sample_size,
            *config.method_config.model.latent_shape,
            device=device,
        )
        x_gen = model.decode(y=z)
        projected = project_to_plane(x_gen, basis, data_config.dataset_config)
        gen_offplane = (
            orthogonal_residual(x_gen, basis, data_config.dataset_config)
            .norm(dim=-1)
            .mean()
            .item()
        )
        gen_radius = projected.norm(dim=-1).mean().item()
        gen_ring_residual = (
            compute_mode_ring_residual(
                points_2d=projected,
                mode_centers=data_config.dataset_config.get_mode_centers().to(
                    device=projected.device, dtype=projected.dtype
                ),
                ring_radius=data_config.dataset_config.get_ring_radius(),
            )
            .mean()
            .item()
        )
    model.train()
    return {
        "gen_offplane": gen_offplane,
        "gen_radius": gen_radius,
        "gen_ring_residual": gen_ring_residual,
    }


def make_generation_figure(
    config: ExperimentConfig,
    data_config: DataConfig,
    val_loader: torch.utils.data.DataLoader,
    model: nn.Module,
    device: torch.device,
    title: str,
):
    model.eval()
    with torch.no_grad():
        batch = next(iter(val_loader)).to(device)
        sample_size = int(batch.x_hd.shape[0])
        z = torch.randn(
            sample_size,
            *config.method_config.model.latent_shape,
            device=device,
        )
        x_gen = model.decode(y=z)
    model.train()
    return data_config.dataset_config.visualize(
        points_by_class={
            "data": batch.x_hd.detach().cpu(),
            "model": x_gen.detach().cpu(),
        },
        title=title,
    )


def train_experiment(
    config: ExperimentConfig,
    data_config: DataConfig,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    model: nn.Module,
    device: torch.device,
) -> dict[str, list[float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=0.01)
    state = config.method_config.initialize_state()
    steps_per_epoch = len(train_loader)

    history: dict[str, list[float]] = {
        "step": [],
        "total_loss": [],
        "drift_loss": [],
        "drift_norm": [],
        "density_ratio": [],
        "gen_offplane": [],
        "gen_radius": [],
        "gen_ring_residual": [],
    }

    progress_bar = tqdm(
        range(1, config.train_epochs + 1),
        desc="training",
        unit="epoch",
    )
    generation_display = None
    for epoch in progress_bar:
        epoch_total_loss = 0.0
        epoch_drift_loss = 0.0
        epoch_drift_norm = 0.0
        epoch_density_ratio = 0.0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            output = state.compute_losses(model=model, batch=batch)
            weighted_loss_terms = dict(output.weighted_loss_terms)
            unweighted_loss_terms = dict(output.unweighted_loss_terms)
            total_loss = sum(weighted_loss_terms.values())

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()

            epoch_total_loss += float(total_loss.item())
            epoch_drift_loss += float(unweighted_loss_terms["drift_matching"].item())
            epoch_drift_norm += float(unweighted_loss_terms["drift_norm"].item())
            epoch_density_ratio += float(unweighted_loss_terms["density_ratio"].item())

        if (
            epoch == 1
            or epoch % config.log_every_epochs == 0
            or epoch == config.train_epochs
        ):
            metrics = collect_metrics(
                config=config,
                data_config=data_config,
                val_loader=val_loader,
                basis=basis,
                model=model,
                device=device,
            )
            history["step"].append(epoch * steps_per_epoch)
            history["total_loss"].append(epoch_total_loss / steps_per_epoch)
            history["drift_loss"].append(epoch_drift_loss / steps_per_epoch)
            history["drift_norm"].append(epoch_drift_norm / steps_per_epoch)
            history["density_ratio"].append(epoch_density_ratio / steps_per_epoch)
            history["gen_offplane"].append(metrics["gen_offplane"])
            history["gen_radius"].append(metrics["gen_radius"])
            history["gen_ring_residual"].append(metrics["gen_ring_residual"])
            progress_bar.set_postfix(
                total=f"{history['total_loss'][-1]:.4f}",
                drift=f"{history['drift_loss'][-1]:.4f}",
                norm=f"{history['drift_norm'][-1]:.4f}",
                ratio=f"{history['density_ratio'][-1]:.4f}",
            )
        if epoch % config.display_every_epochs == 0:
            figure = make_generation_figure(
                config=config,
                data_config=data_config,
                val_loader=val_loader,
                model=model,
                device=device,
                title=f"generation progress epoch {epoch}",
            )
            if generation_display is None:
                generation_display = display(figure, display_id=True)
            else:
                generation_display.update(figure)

    return history


def make_snapshot(
    config: ExperimentConfig,
    val_loader: torch.utils.data.DataLoader,
    model: nn.Module,
    device: torch.device,
) -> dict[str, torch.Tensor]:
    model.eval()
    with torch.no_grad():
        batch = next(iter(val_loader)).to(device)
        sample_size = int(batch.x_hd.shape[0])
        z = torch.randn(
            sample_size,
            *config.method_config.model.latent_shape,
            device=device,
        )
        x_gen_hd = model.decode(y=z)
    model.train()
    return {
        "x_true_hd": batch.x_hd.detach().cpu(),
        "x_gen_hd": x_gen_hd.detach().cpu(),
    }


def plot_training_curves(history: dict[str, list[float]]) -> None:
    steps = history["step"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

    axes[0, 0].plot(steps, history["total_loss"], label="total")
    axes[0, 0].plot(steps, history["drift_loss"], label="drift")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("steps")
    axes[0, 0].legend()

    axes[0, 1].plot(steps, history["drift_norm"])
    axes[0, 1].set_title("Drift Norm")
    axes[0, 1].set_xlabel("steps")

    axes[1, 0].plot(steps, history["density_ratio"])
    axes[1, 0].set_title("Density Ratio")
    axes[1, 0].set_xlabel("steps")

    axes[1, 1].plot(steps, history["gen_offplane"], label="offplane")
    axes[1, 1].plot(steps, history["gen_ring_residual"], label="ring residual")
    axes[1, 1].plot(steps, history["gen_radius"], label="radius")
    axes[1, 1].set_title("Geometry")
    axes[1, 1].set_xlabel("steps")
    axes[1, 1].legend()

    fig.tight_layout()
    plt.show()


def run_single_experiment(
    config: ExperimentConfig,
    data_config: DataConfig,
) -> dict[str, object]:
    set_seed(SEED)
    basis = data_config.dataset_config.get_basis().to(DEVICE)
    train_loader = data_config.get_trainloader()
    val_loader = data_config.get_valloader()
    model = config.method_config.get_model().to(DEVICE)
    history = train_experiment(
        config=config,
        data_config=data_config,
        train_loader=train_loader,
        val_loader=val_loader,
        basis=basis,
        model=model,
        device=DEVICE,
    )
    snapshot = make_snapshot(
        config=config,
        val_loader=val_loader,
        model=model,
        device=DEVICE,
    )
    metrics = collect_metrics(
        config=config,
        data_config=data_config,
        val_loader=val_loader,
        basis=basis,
        model=model,
        device=DEVICE,
    )
    return {
        "config": config,
        "data_config": data_config,
        "model": model,
        "history": history,
        "snapshot": snapshot,
        "metrics": metrics,
    }

In [ ]:
set_seed(SEED)
preview_model = config.method_config.get_model()
preview_model.eval()
with torch.no_grad():
    preview_latents = torch.randn(
        int(preview_batch.x_hd.shape[0]),
        *config.method_config.model.latent_shape,
    )
    preview_pushforward = preview_model.decode(y=preview_latents).detach().cpu()

preview_figure = dataset_config.visualize(
    points_by_class={
        "data": preview_batch.x_hd.detach().cpu(),
        "model": preview_pushforward,
    },
)
preview_figure


In [ ]:
results = run_single_experiment(config=config, data_config=data_config)
print(results["metrics"])


In [ ]:
plot_training_curves(results["history"])

results["data_config"].dataset_config.visualize(
    points_by_class={
        "data": results["snapshot"]["x_true_hd"],
        "model": results["snapshot"]["x_gen_hd"],
    },
).show(config={"scrollZoom": True})
